## Configuração do ambiente (local)

**Objetivo:** Treinar CNN customizada e ResNet50 (Transfer Learning) nos mesmos 5 folds dos baselines.

**Entradas:** `Data/processed/dataset_abide_processado.pkl` (07), `Data/conectivity/connectivity_matrices_abide_validado.pkl` (05), `Results/08_resultados_baselines.csv` (08, opcional).

**Processamento:** Matrizes 200×200 reescaladas para [0, 1]; EarlyStopping e ReduceLROnPlateau; GPU usada se disponível.

**Saídas:** `Models/09_cnn_customizada_fold{0..4}.keras`, `Models/09_resnet50_transfer_fold{0..4}.keras`, `Results/09_comparacao_final_todos_modelos.csv`.

In [1]:
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

Projeto (BASE_DIR) = C:\Users\Maria Eduarda\Downloads\TCC_TEA


## Verificando a GPU

In [2]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used',
                            '--format=csv,noheader'], capture_output=True, text=True, check=True)
        print(r.stdout.strip())
        print('GPU NVIDIA detectada pelo driver (nvidia-smi).')
    except Exception as e:
        print('nvidia-smi encontrado, mas falhou:', e)
else:
    print('nvidia-smi não encontrado — sem driver NVIDIA; o TensorFlow executará em CPU.')

NVIDIA GeForce GTX 1650, 4096 MiB, 760 MiB
GPU NVIDIA detectada pelo driver (nvidia-smi).


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reprodutibilidade (adicionado na migração): fixa as sementes do TensorFlow/Keras.
# Não ativa "op determinism" para não alterar o desempenho; o treino em GPU pode
# variar levemente entre execuções.
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs visíveis para o TensorFlow:', gpus)

if gpus:
    print('GPU detectada')
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Precisão mista (mixed_float16) ativada.')
else:
    print('GPU não detectada — executando em CPU')
    print('Mantendo float32 padrão (mixed_float16 não traz benefício em CPU).')

TensorFlow: 2.21.0
GPUs visíveis para o TensorFlow: []
GPU não detectada — executando em CPU
Mantendo float32 padrão (mixed_float16 não traz benefício em CPU).


## Preparação dos dados: da matriz de conectividade ao tensor de entrada
Diferente do notebook 08 (que usava o vetor achatado de 19.900 features), aqui usamos a **matriz completa 200×200**, porque é isso que dá à CNN a estrutura espacial 2D que ela precisa para funcionar. Duas transformações são necessárias:

1. **Réplica de canal:** a ResNet50 espera entrada com 3 canais (RGB). Nossa matriz tem 1 canal (a correlação). Replicamos o mesmo valor nos 3 canais — não inventa informação nova, apenas satisfaz o formato de entrada esperado pela rede pré-treinada.
2. **Reescala para `[0, 1]`:** a correlação de Pearson já está em `[-1, 1]`; aplicamos `(x + 1) / 2` para levar ao intervalo `[0, 1]`, mais estável numericamente para redes neurais (evita gradientes muito grandes/pequenos nas primeiras camadas) e compatível com a normalização que o `preprocess_input` das redes pré-treinadas do Keras espera receber.

**Importante — mesma ordem de sujeitos que os folds do notebook 07:** os folds salvos em `dataset_abide_processado.pkl` são índices posicionais (`train_idx`, `test_idx`) que só são válidos se construirmos o array de imagens **na mesma ordem de `sub_ids`** que o notebook 07 usou. Por isso carregamos `dataset['sub_ids']` (salvo junto com os folds) e indexamos `conn_dict` exatamente nessa ordem, em vez de usar `list(conn_dict.keys())` de novo (que poderia, em teoria, vir em outra ordem se o dicionário fosse reconstruído de forma diferente).

In [4]:
exigir_arquivo(PATHS['conectivity'] / 'connectivity_matrices_abide_validado.pkl', '05_Validacao_ABIDE.ipynb')
exigir_arquivo(PATHS['processed'] / 'dataset_abide_processado.pkl', '07_Preparacao_Dados_ABIDE.ipynb')

import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.applications.resnet50 import preprocess_input

with open(os.path.join(PATHS['conectivity'], 'connectivity_matrices_abide_validado.pkl'), 'rb') as f:
    conn_dict = pickle.load(f)

with open(os.path.join(PATHS['processed'], 'dataset_abide_processado.pkl'), 'rb') as f:
    dataset = pickle.load(f)

sub_ids = dataset['sub_ids']   # MESMA ordem usada para gerar os folds no notebook 07
y = dataset['y']
groups = dataset['groups']
folds = dataset['folds']

# Matriz completa (N, 200, 200), na mesma ordem de sub_ids/y/groups/folds
matrices = np.array([conn_dict[s] for s in sub_ids]).astype('float32')
print('Matrizes:', matrices.shape)

# Reescala de [-1, 1] para [0, 1]
matrices_01 = (matrices + 1.0) / 2.0

# Réplica de canal: (N, 200, 200) -> (N, 200, 200, 3), para a ResNet50
X_img_3ch = np.repeat(matrices_01[..., np.newaxis], 3, axis=-1)
# Pré-processamento específico da ResNet50
X_resnet = preprocess_input(
    X_img_3ch * 255.0
).astype('float32')

# Versão de 1 canal, para a CNN customizada (não precisa de 3 canais)
X_img_1ch = matrices_01[..., np.newaxis]

print('X_img_3ch:', X_img_3ch.shape, '| X_img_1ch:', X_img_1ch.shape)
print('Distribuição y:', np.bincount(y))

Matrizes: (871, 200, 200)
X_img_3ch: (871, 200, 200, 3) | X_img_1ch: (871, 200, 200, 1)
Distribuição y: [468 403]


## CNN Customizada — arquitetura e teoria

Uma CNN pequena, treinada do zero, serve de **ablação**: mostra o que uma rede "sem conhecimento prévio" consegue extrair só com os ~870 exemplos que temos, antes de comparar com a versão que usa transfer learning.

**Blocos da arquitetura:**
- **Conv2D + BatchNorm + ReLU**, repetido 3 vezes com número crescente de filtros (32→64→128) — cada bloco aprende padrões progressivamente mais abstratos; o BatchNorm estabiliza o treinamento normalizando as ativações a cada mini-lote, reduzindo sensibilidade à escolha de taxa de aprendizado.
- **MaxPooling2D** após cada bloco — reduz a dimensionalidade espacial progressivamente (de 200×200 para dimensões menores), tornando os filtros seguintes mais robustos a pequenos deslocamentos espaciais e reduzindo custo computacional.
- **GlobalAveragePooling2D** no final, em vez de `Flatten` — reduz drasticamente o número de parâmetros da camada densa final (que seria enorme com `Flatten` em um mapa de features ainda grande), o que é uma forma direta de mitigar overfitting num dataset pequeno.
- **Dropout** antes da camada de saída — desativa aleatoriamente uma fração das unidades durante o treino, forçando a rede a não depender excessivamente de nenhum neurônio específico (outra defesa direta contra overfitting).
- **Dense(1, sigmoid)** — saída binária (TEA vs. controle).

In [5]:
def build_custom_cnn(input_shape=(200, 200, 1)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2),

        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid', dtype='float32'),  # float32 na saída: estabilidade com mixed_float16
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

build_custom_cnn().summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 200, 200, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 200, 200, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 200, 200, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 100, 100, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 100, 100, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 100, 100, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 100, 100, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 50, 50, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 50, 50, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 50, 50, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 50, 50, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 25, 25, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,889 (398.00 KB)

 Trainable params: 101,441 (396.25 KB)

 Non-trainable params: 448 (1.75 KB)

## Treinamento da CNN customizada — mesmos 5 folds do notebook 08

Usamos os mesmos folds (`StratifiedGroupKFold` por site) do notebook 07/08 — é isso que torna a comparação entre baselines e deep learning **metodologicamente válida**: todos os modelos são avaliados exatamente nos mesmos sujeitos de treino/teste em cada fold, então a diferença de desempenho reflete o modelo, não uma diferença de partição dos dados.

**`EarlyStopping`** interrompe o treino quando a métrica de validação para de melhorar por algumas épocas (`patience`), evitando treinar além do ponto de overfitting e economizando tempo de GPU. **`ReduceLROnPlateau`** reduz a taxa de aprendizado quando o progresso estagna, permitindo ajustes mais finos nos pesos perto da convergência.

Diferente do notebook 08 (que fez busca exaustiva de hiperparâmetros via `GridSearchCV`), aqui usamos uma arquitetura e hiperparâmetros **fixos e razoáveis**, sem grid search — buscar hiperparâmetros de deep learning exaustivamente exigiria treinar dezenas de redes por fold, o que é computacionalmente desproporcional ao escopo de um TCC. Essa é uma limitação documentada, não um descuido: fica registrada explicitamente para o Capítulo 4.

In [6]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
import time

resultados_cnn = []
melhores_modelos_cnn = {}

BATCH_SIZE = 16
EPOCHS_MAX = 60

for i, (train_idx, test_idx) in enumerate(folds):
    t0 = time.time()
    print(f'\n=== CNN Customizada — Fold {i} ===')

    # ============================================================
    # 1. Separação do fold externo
    # ============================================================
    # O conjunto de teste permanece totalmente separado.
    X_train_full = X_img_1ch[train_idx]
    y_train_full = y[train_idx]
    groups_train_full = groups[train_idx]

    X_test = X_img_1ch[test_idx]
    y_test = y[test_idx]

    # ============================================================
    # 2. Validação interna respeitando SITE_ID
    # ============================================================
    # O validation_split do Keras foi removido porque ele não
    # considera os grupos (SITE_ID).
    inner_cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    inner_train_idx, val_idx = next(
        inner_cv.split(
            X_train_full,
            y_train_full,
            groups_train_full
        )
    )

    X_train = X_train_full[inner_train_idx]
    y_train = y_train_full[inner_train_idx]

    X_val = X_train_full[val_idx]
    y_val = y_train_full[val_idx]

    print(f'Treino: {X_train.shape}')
    print(f'Validação: {X_val.shape}')
    print(f'Teste: {X_test.shape}')

    # ============================================================
    # 3. Criação do modelo
    # ============================================================
    modelo = build_custom_cnn(
        input_shape=(200, 200, 1)
    )

    # ============================================================
    # 4. Callbacks
    # ============================================================
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            min_lr=1e-6
        ),
    ]

    # ============================================================
    # 5. Treinamento
    # ============================================================
    history = modelo.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_MAX,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )

    # ============================================================
    # 6. Avaliação no TESTE externo
    # ============================================================
    y_proba = modelo.predict(
        X_test,
        verbose=0
    ).ravel()

    y_pred = (
        y_proba >= 0.5
    ).astype(int)

    acc = accuracy_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_proba
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    tempo_fold = time.time() - t0

    # ============================================================
    # 7. Armazenamento dos resultados
    # ============================================================
    resultados_cnn.append({
        'fold': i,
        'accuracy': acc,
        'f1': f1,
        'auc_roc': auc,
        'epocas_treinadas': len(
            history.history['loss']
        ),
        'tempo_s': tempo_fold
    })

    print(
        f'  acc={acc:.3f} '
        f'f1={f1:.3f} '
        f'auc={auc:.3f} | '
        f'{len(history.history["loss"])} épocas | '
        f'{tempo_fold:.1f}s'
    )

    print(
        '  Matriz de confusão:\n',
        cm
    )

    # ============================================================
    # 8. Salvamento do modelo
    # ============================================================
    modelo_path = os.path.join(
        PATHS['models'],
        f'09_cnn_customizada_fold{i}.keras'
    )

    modelo.save(modelo_path)

    melhores_modelos_cnn[
        f'CNN_fold{i}'
    ] = modelo_path


# ================================================================
# 9. Resumo dos resultados
# ================================================================
df_cnn = pd.DataFrame(
    resultados_cnn
)

print(
    '\n=== Resultado CNN Customizada '
    '(média entre folds) ==='
)

print(
    df_cnn[
        [
            'fold',
            'accuracy',
            'f1',
            'auc_roc',
            'tempo_s'
        ]
    ]
)

print(
    f"Accuracy média: "
    f"{df_cnn['accuracy'].mean():.4f} "
    f"± {df_cnn['accuracy'].std():.4f}"
)


=== CNN Customizada — Fold 0 ===
Treino: (763, 200, 200, 1)
Validação: (67, 200, 200, 1)
Teste: (41, 200, 200, 1)
  acc=0.537 f1=0.698 auc=0.483 | 12 épocas | 287.4s
  Matriz de confusão:
 [[ 0 19]
 [ 0 22]]

=== CNN Customizada — Fold 1 ===
Treino: (569, 200, 200, 1)
Validação: (129, 200, 200, 1)
Teste: (173, 200, 200, 1)
  acc=0.474 f1=0.486 auc=0.473 | 21 épocas | 354.3s
  Matriz de confusão:
 [[39 62]
 [29 43]]

=== CNN Customizada — Fold 2 ===
Treino: (639, 200, 200, 1)
Validação: (67, 200, 200, 1)
Teste: (165, 200, 200, 1)
  acc=0.448 f1=0.619 auc=0.517 | 14 épocas | 270.9s
  Matriz de confusão:
 [[ 0 91]
 [ 0 74]]

=== CNN Customizada — Fold 3 ===
Treino: (433, 200, 200, 1)
Validação: (114, 200, 200, 1)
Teste: (324, 200, 200, 1)
  acc=0.460 f1=0.630 auc=0.473 | 9 épocas | 120.3s
  Matriz de confusão:
 [[  0 175]
 [  0 149]]

=== CNN Customizada — Fold 4 ===
Treino: (670, 200, 200, 1)
Validação: (33, 200, 200, 1)
Teste: (168, 200, 200, 1)
  acc=0.488 f1=0.000 auc=0.490 | 9 época

## Transfer Learning com ResNet50 — arquitetura e teoria

**Congelar vs. ajustar (fine-tuning):** a ResNet50 pré-treinada tem ~23 milhões de parâmetros. Se treinássemos todos eles com nossos ~870 exemplos, seria pior que a CNN customizada (muito mais parâmetros para muito menos dados — overfitting quase garantido). A estratégia padrão de transfer learning é:

1. **Congelar (`trainable=False`)** todas as camadas convolucionais da ResNet50 pré-treinada — mantemos os filtros aprendidos no ImageNet exatamente como estão.
2. **Adicionar uma "cabeça" (head) nova e pequena** por cima: `GlobalAveragePooling2D` + `Dense` — só essas poucas camadas novas são treinadas, com muito menos parâmetros livres, ajustadas especificamente para distinguir TEA de controle a partir dos mapas de features já extraídos pela ResNet50.

Essa divisão (base congelada + cabeça treinável) é o que torna transfer learning viável mesmo com um dataset pequeno: o número efetivo de parâmetros sendo *aprendidos* aqui é ordens de magnitude menor que treinar a rede inteira do zero.

In [7]:
def build_transfer_model(input_shape=(200, 200, 3)):
    base = applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
    )
    base.trainable = False  # congela a base pré-treinada

    modelo = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid', dtype='float32'),
    ])
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return modelo

build_transfer_model().summary()


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 10s 0us/step


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │       131,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,718,913 (90.48 MB)

 Trainable params: 131,201 (512.50 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [8]:
from sklearn.model_selection import StratifiedGroupKFold
resultados_transfer = []
melhores_modelos_transfer = {}

for i, (train_idx, test_idx) in enumerate(folds):
    t0 = time.time()
    print(
        f'\n=== ResNet50 (Transfer Learning) — Fold {i} ==='
    )

    # ============================================================
    # 1. Separação do fold externo
    # ============================================================
    X_train_full = X_resnet[train_idx]
    y_train_full = y[train_idx]
    groups_train_full = groups[train_idx]

    X_test = X_resnet[test_idx]
    y_test = y[test_idx]

    # ============================================================
    # 2. Validação interna respeitando SITE_ID
    # ============================================================
    inner_cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    inner_train_idx, val_idx = next(
        inner_cv.split(
            X_train_full,
            y_train_full,
            groups_train_full
        )
    )

    X_train = X_train_full[inner_train_idx]
    y_train = y_train_full[inner_train_idx]

    X_val = X_train_full[val_idx]
    y_val = y_train_full[val_idx]

    print(
        f'Treino: {X_train.shape}'
    )

    print(
        f'Validação: {X_val.shape}'
    )

    print(
        f'Teste: {X_test.shape}'
    )

    # ============================================================
    # 3. Criação da ResNet50
    # ============================================================
    modelo = build_transfer_model(
        input_shape=(200, 200, 3)
    )

    # ============================================================
    # 4. Callbacks
    # ============================================================
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            min_lr=1e-6
        ),
    ]

    # ============================================================
    # 5. Treinamento
    # ============================================================
    history = modelo.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_MAX,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )

    # ============================================================
    # 6. Avaliação no TESTE externo
    # ============================================================
    y_proba = modelo.predict(
        X_test,
        verbose=0
    ).ravel()

    y_pred = (
        y_proba >= 0.5
    ).astype(int)

    acc = accuracy_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_proba
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    tempo_fold = time.time() - t0

    # ============================================================
    # 7. Armazenamento dos resultados
    # ============================================================
    resultados_transfer.append({
        'fold': i,
        'accuracy': acc,
        'f1': f1,
        'auc_roc': auc,
        'epocas_treinadas': len(
            history.history['loss']
        ),
        'tempo_s': tempo_fold
    })

    print(
        f'  acc={acc:.3f} '
        f'f1={f1:.3f} '
        f'auc={auc:.3f} | '
        f'{len(history.history["loss"])} épocas | '
        f'{tempo_fold:.1f}s'
    )

    print(
        '  Matriz de confusão:\n',
        cm
    )

    # ============================================================
    # 8. Salvamento do modelo
    # ============================================================
    modelo_path = os.path.join(
        PATHS['models'],
        f'09_resnet50_transfer_fold{i}.keras'
    )

    modelo.save(modelo_path)

    melhores_modelos_transfer[
        f'ResNet50_fold{i}'
    ] = modelo_path


# ================================================================
# 9. Resumo dos resultados
# ================================================================
df_transfer = pd.DataFrame(
    resultados_transfer
)

print(
    '\n=== Resultado ResNet50 Transfer Learning '
    '(média entre folds) ==='
)

print(
    df_transfer[
        [
            'fold',
            'accuracy',
            'f1',
            'auc_roc',
            'tempo_s'
        ]
    ]
)

print(
    f"Accuracy média: "
    f"{df_transfer['accuracy'].mean():.4f} "
    f"± {df_transfer['accuracy'].std():.4f}"
)


=== ResNet50 (Transfer Learning) — Fold 0 ===
Treino: (763, 200, 200, 3)
Validação: (67, 200, 200, 3)
Teste: (41, 200, 200, 3)
  acc=0.585 f1=0.638 auc=0.500 | 9 épocas | 195.3s
  Matriz de confusão:
 [[ 9 10]
 [ 7 15]]

=== ResNet50 (Transfer Learning) — Fold 1 ===
Treino: (569, 200, 200, 3)
Validação: (129, 200, 200, 3)
Teste: (173, 200, 200, 3)
  acc=0.514 f1=0.580 auc=0.624 | 21 épocas | 436.2s
  Matriz de confusão:
 [[31 70]
 [14 58]]

=== ResNet50 (Transfer Learning) — Fold 2 ===
Treino: (639, 200, 200, 3)
Validação: (67, 200, 200, 3)
Teste: (165, 200, 200, 3)
  acc=0.515 f1=0.167 auc=0.534 | 21 épocas | 401.2s
  Matriz de confusão:
 [[77 14]
 [66  8]]

=== ResNet50 (Transfer Learning) — Fold 3 ===
Treino: (433, 200, 200, 3)
Validação: (114, 200, 200, 3)
Teste: (324, 200, 200, 3)
  acc=0.549 f1=0.131 auc=0.550 | 9 épocas | 145.5s
  Matriz de confusão:
 [[167   8]
 [138  11]]

=== ResNet50 (Transfer Learning) — Fold 4 ===
Treino: (670, 200, 200, 3)
Validação: (33, 200, 200, 3)
Te

## Consolidação: comparando com os baselines do notebook 08

Junta os resultados da CNN customizada e da ResNet50 com os resultados salvos pelo notebook 08 (SVM, RF, XGBoost), numa única tabela — a comparação final que responde à pergunta central deste notebook: *deep learning supera os baselines clássicos, dado o tamanho da nossa amostra?*

In [9]:
def resumir(nome, lista_resultados):
    df = pd.DataFrame(lista_resultados)
    return {
        'modelo': nome,
        'accuracy_media': df['accuracy'].mean(), 'accuracy_std': df['accuracy'].std(),
        'f1_media': df['f1'].mean(), 'f1_std': df['f1'].std(),
        'auc_roc_media': df['auc_roc'].mean(), 'auc_roc_std': df['auc_roc'].std(),
    }

resumo_dl = pd.DataFrame([
    resumir('CNN_Customizada', resultados_cnn),
    resumir('ResNet50_TransferLearning', resultados_transfer),
])

baselines_path = os.path.join(PATHS['results'], '08_resultados_baselines.csv')
if os.path.exists(baselines_path):
    df_baselines = pd.read_csv(baselines_path)
    df_comparacao = pd.concat([df_baselines, resumo_dl], ignore_index=True)
else:
    print('Aviso: 08_resultados_baselines.csv não encontrado — comparando só os modelos deste notebook.')
    df_comparacao = resumo_dl

print('=== Comparação final: baselines (notebook 08) vs. deep learning (notebook 09) ===')
print(df_comparacao)

df_comparacao.to_csv(os.path.join(PATHS['results'], '09_comparacao_final_todos_modelos.csv'), index=False)


=== Comparação final: baselines (notebook 08) vs. deep learning (notebook 09) ===
                      modelo  accuracy_media  accuracy_std  f1_media  \
0                        SVM        0.662196      0.067883  0.622914   
1               RandomForest        0.599616      0.033523  0.476399   
2                    XGBoost        0.631313      0.063720  0.581197   
3            CNN_Customizada        0.481406      0.034249  0.486711   
4  ResNet50_TransferLearning        0.540013      0.029303  0.350242   

     f1_std  auc_roc_media  auc_roc_std  
0  0.095912       0.719842     0.046182  
1  0.057632       0.678973     0.064906  
2  0.098858       0.679641     0.053639  
3  0.282758       0.487312     0.018292  
4  0.240190       0.550645     0.045568  
